# MaskFormer Inference Pipeline

Load a trained `GLCAnomalyMaskFormer` checkpoint, run inference on welding robot data, and produce dual-criterion anomaly verdicts.

**Pipeline:**
1. Load trained model checkpoint
2. Load CSV data → macro-windows (same preprocessing as training)
3. Run per-file inference via `compute_combined_heatmap` + `detect_anomaly`
4. Fit percentile threshold via `MaskFormerThresholdCreator` on normal files
5. Apply threshold + per-sensor verdicts via `MaskFormerApplier`
6. Visualize heatmaps, anomaly scores, and localization

In [ ]:
# %% Imports
import sys
import csv
import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Add project root to path
project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.anomaly_detection.models import (
    GLCAnomalyMaskFormer,
    create_model,
    filter_model_kwargs,
    MaskFormerModels,
)
# Local (Spark-free) 3-class inference pipeline
from app.anomaly_detection.inference.maskformer.local import (
    MaskFormerLocalPreResult,
    MaskFormerLocalThresholdCreator,
    MaskFormerLocalApplier,
)
from app.anomaly_detection.training.datasets import RawWindowDataset

print("All imports OK")
print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")


## 1. Load Trained Model Checkpoint

Load the model state_dict + config saved by the training notebook.

In [ ]:
# Path to trained checkpoint from Training-MaskFormer notebook
CKPT_DIR = project_root / "artifacts" / "maskformer_checkpoint"
MODEL_SLUG = "GLCAnomalyMaskFormer"
MODEL_FILE = f"model_{MODEL_SLUG}_best.pt"
META_FILE = f"metadata_{MODEL_SLUG}_best.json"
EXTRA_FILE = CKPT_DIR / "training_extra.json"

if not (CKPT_DIR / MODEL_FILE).exists():
    print(f"Checkpoint not found at {CKPT_DIR / MODEL_FILE}")
    print("Run the Training-MaskFormer notebook first to train and save a model.")
else:
    # Reconstruct model using BaseAnomalyDetectionModel.load_from_checkpoint
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GLCAnomalyMaskFormer.load_from_checkpoint(
        checkpoint=str(CKPT_DIR),
        model_file=MODEL_FILE,
        metadata_file=META_FILE,
        device=str(device),
    )
    print(f"Model restored to {device}: {model.name}")
    print(f"  Params: {sum(p.numel() for p in model.parameters()):,}")

    # Load training extras (history, codebooks) if available
    extras = {}
    if EXTRA_FILE.exists():
        with open(EXTRA_FILE) as f:
            extras = json.load(f)
        print(f"  Training extras loaded: history ({len(extras.get('history',{}).get('loss',[]))} epochs), "
              f"robot_codebook ({len(extras.get('robot_codebook',{}))} codes)")


## 2. Load and Preprocess CSV Data

Same macro-window preprocessing as the training notebook:
- Group by `(robot_code, start_time, program_number)`
- Sort by `serial_number`
- Pad or slice to `seq_len=240`
- Build robot/program codebooks

In [ ]:
CSV_PATH = project_root / "data" / "mock_welding_robot_data.csv"
SENSOR_COLS = ["wire_feed_speed_mm_min", "welding_voltage_v", "welding_current_a"]
SEQ_LEN = 240

print(f"Reading {CSV_PATH}...")
with open(CSV_PATH, "r") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"Total rows: {len(rows):,}")

# Group rows by (robot_code, start_time, program_number)
groups = defaultdict(list)
for r in rows:
    key = (r["robot_code"], r["start_time"], r["program_number"])
    groups[key].append(r)

print(f"Total files (groups): {len(groups)}")

# Build codebooks
all_robot_codes = sorted(set(r["robot_code"] for r in rows))
all_program_nums = sorted(set(r["program_number"] for r in rows))
robot_to_idx = {code: i for i, code in enumerate(all_robot_codes)}
program_to_idx = {pn: i for i, pn in enumerate(all_program_nums)}

# Build file-windows
file_records = []
for key, group_rows in groups.items():
    group_rows.sort(key=lambda x: int(x["serial_number"]))
    file_arr = np.array(
        [[float(r[col]) for col in SENSOR_COLS] for r in group_rows],
        dtype=np.float32,
    )
    T = file_arr.shape[0]
    if T >= SEQ_LEN:
        window = file_arr[:SEQ_LEN, :].T  # [C, T]
    else:
        pad = np.zeros((SEQ_LEN - T, 3), dtype=np.float32)
        window = np.vstack([file_arr, pad]).T

    file_records.append({
        "key": key,
        "window": window,
        "robot_idx": robot_to_idx[key[0]],
        "program_idx": program_to_idx[key[1]],
        "robot_code": key[0],
        "program_number": int(key[1]),
        "start_time": key[2],
    })

print(f"Preprocessed {len(file_records)} file-windows")
print(f"Example: robot={file_records[0]['robot_code']}, "
      f"program={file_records[0]['program_number']}, "
      f"window shape={file_records[0]['window'].shape}")


## 3. Run Inference Per File

Use the model's `detect_anomaly()` to get:
- `anomaly_score` — file-level score (max of combined heatmap)
- `heatmap` — combined [sensors × patches] heatmap
- `assoc_heatmap` — association-discrepancy only
- `recon_heatmap` — reconstruction-only

In [ ]:
# Run inference on all files with a progress indicator
results = []

for i, rec in enumerate(file_records):
    inp = torch.from_numpy(rec["window"]).unsqueeze(0).to(device)  # [1, C, T]
    robot_t = torch.tensor([rec["robot_idx"]], dtype=torch.long, device=device)
    prog_t = torch.tensor([rec["program_idx"]], dtype=torch.long, device=device)

    with torch.no_grad():
        out = model.detect_anomaly(
            raw_file=inp,
            robot_idx=robot_t,
            program_idx=prog_t,
            fast=True,
        )

    results.append({
        "robot_code": rec["robot_code"],
        "program_number": rec["program_number"],
        "start_time": rec["start_time"],
        "anomaly_score": out["anomaly_score"],
        "is_anomaly": out["is_anomaly"],
        "heatmap": out["heatmap"].numpy(),          # [C, P]
        "assoc_heatmap": out["assoc_heatmap"].numpy(),
        "recon_heatmap": out["recon_heatmap"].numpy(),
    })

    if (i + 1) % max(1, len(file_records) // 5) == 0:
        print(f"  [{i+1}/{len(file_records)}] processed")

print(f"\nDone. {len(results)} files scored.")

# Score statistics
scores = np.array([r["anomaly_score"] for r in results])
print(f"\nAnomaly score distribution:")
print(f"  Min:    {scores.min():.6f}")
print(f"  Max:    {scores.max():.6f}")
print(f"  Mean:   {scores.mean():.6f}")
print(f"  Median: {np.median(scores):.6f}")
print(f"  P95:    {np.percentile(scores, 95):.6f}")
print(f"  P99:    {np.percentile(scores, 99):.6f}")


## 4. 3-Class Local Inference Pipeline

Use `MaskFormerLocalPreResult.score_batch() → MaskFormerLocalThresholdCreator().fit() → MaskFormerLocalApplier().apply()`.

The `score_batch` accepts `list[dict]` records with windows + serials, simulates the macro-window accumulation logic that the Spark version would use, and returns per-file pre-results with per-sensor heatmaps.

In [ ]:
# Build macro-window records for the local pre-result pipeline
# Each record simulates a file with multiple overlapping windows
pre_result_records = []
for rec in file_records:
    window = rec["window"]  # [C, T]
    C, T = window.shape
    # Create 3 overlapping windows per file (offset by patch_stride=16)
    patch_stride = 16
    windows_list = []
    serials_list = []
    for w in range(0, min(T - SEQ_LEN + 1, 3) * patch_stride, patch_stride):
        if w + SEQ_LEN <= T:
            windows_list.append(window[:, w:w+SEQ_LEN])
            serials_list.append(w)
    if not windows_list:
        windows_list.append(window[:, :SEQ_LEN])
        serials_list.append(0)
    pre_result_records.append({
        "robot_code": rec["robot_code"],
        "program_number": rec["program_number"],
        "start_time": rec["start_time"],
        "windows": np.stack(windows_list),  # [N, C, W]
        "serials": np.array(serials_list, dtype=np.int64),
    })

print(f"Prepared {len(pre_result_records)} records for local pipeline")
print(f"Example windows shape: {pre_result_records[0]['windows'].shape}")

# Step 1: Score batch via MaskFormerLocalPreResult
metadata = model.get_metadata()
state_dict = model.state_dict()
pre_result = MaskFormerLocalPreResult(
    model_version=MaskFormerModels.GLC_ANOMALY_MASKFORMER,
    metadata=metadata,
    state_dict=state_dict,
    quantile=0.95,
    k_skip=0,
    fast=True,
    device=str(device),
)
pre_scores = pre_result.score_batch(pre_result_records)

# Step 2: Fit threshold via MaskFormerLocalThresholdCreator
threshold_creator = MaskFormerLocalThresholdCreator(quantile=0.95)
threshold_creator.fit_from_pre_results(pre_scores)

global_threshold = threshold_creator.threshold
print(f"Global threshold (P95): {global_threshold:.6f}")
print(f"Threshold config: {threshold_creator.to_dict()}")

# Step 3: Apply thresholds via MaskFormerLocalApplier
applier = MaskFormerLocalApplier(
    threshold_config={"global_threshold": global_threshold},
    quantile=0.95,
)
applied_results = applier.apply(pre_scores)

n_anomaly = sum(1 for r in applied_results if r["is_file_anomaly"])
n_normal = len(applied_results) - n_anomaly
print(f"\nVerdict distribution (via local pipeline):")
print(f"  Normal files:    {n_normal} ({100*n_normal/len(applied_results):.1f}%)")
print(f"  Anomalous files: {n_anomaly} ({100*n_anomaly/len(applied_results):.1f}%)")

# Show top-5 most anomalous files
sorted_results = sorted(applied_results, key=lambda r: float(r[f"p95_score"]), reverse=True)
print(f"\nTop-5 most anomalous files:")
for i, r in enumerate(sorted_results[:5]):
    print(f"  #{i+1}: robot={r['robot_code']}, prog={r['program_number']}, "
          f"score={r['p95_score']:.6f} {'⚠️' if r['is_file_anomaly'] else '✓'}")


## 5. Per-Sensor Heatmap Analysis (MaskFormerPreResult-style)

Compute per-sensor heatmaps and p95 scores as `MaskFormerPreResult` would, then apply per-sensor thresholds.

In [ ]:
SENSOR_NAMES = ["wire_feed_speed", "welding_voltage", "welding_current"]

# For each file, compute per-sensor max heatmap score
per_sensor_max = {s: [] for s in SENSOR_NAMES}
for r in results:
    hm = r["heatmap"]  # [3, P]
    for si, sn in enumerate(SENSOR_NAMES):
        per_sensor_max[sn].append(hm[si].max())

print("Per-sensor max heatmap statistics:")
for sn in SENSOR_NAMES:
    vals = np.array(per_sensor_max[sn])
    print(f"  {sn}: mean={vals.mean():.4f}, max={vals.max():.4f}, P95={np.percentile(vals, 95):.4f}")

# Build per-sensor thresholds (P95 per sensor)
sensor_thresholds = {
    sn: float(np.percentile(per_sensor_max[sn], 95))
    for sn in SENSOR_NAMES
}
print(f"\nPer-sensor P95 thresholds:")
for sn, th in sensor_thresholds.items():
    print(f"  {sn}: {th:.6f}")

# Apply per-sensor verdicts
for r in results:
    r["sensor_verdicts"] = {}
    hm = r["heatmap"]
    for si, sn in enumerate(SENSOR_NAMES):
        r["sensor_verdicts"][sn] = bool(hm[si].max() > sensor_thresholds[sn])
    r["is_file_anomaly"] = r["anomaly_score"] > global_threshold

# Show sample verdicts
print(f"\nSample verdicts (first 10 files):")
print(f"{'robot':>6} {'prog':>5} {'score':>10} {'file?':>6} ", end="")
for sn in SENSOR_NAMES:
    print(f"{sn[:8]:>8}", end="")
print()
for r in results[:10]:
    print(f"{r['robot_code']:>6} {r['program_number']:>5} {r['anomaly_score']:>10.6f} "
          f"{'⚠️' if r['is_file_anomaly'] else '✓':>6} ", end="")
    for sn in SENSOR_NAMES:
        print(f"{'⚠️' if r['sensor_verdicts'][sn] else '✓':>8}", end="")
    print()


## 6. Visualize Results

### 6a. Anomaly Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

scores = np.array([r["anomaly_score"] for r in results])
is_anom = np.array([r["is_file_anomaly"] for r in results])

# Histogram
ax = axes[0]
ax.hist(scores[~is_anom], bins=40, alpha=0.7, label=f"Normal ({n_normal})", color="steelblue")
ax.hist(scores[is_anom], bins=20, alpha=0.7, label=f"Anomaly ({n_anomaly})", color="crimson")
ax.axvline(global_threshold, color="red", linestyle="--", label=f"P95={global_threshold:.4f}")
ax.set_xlabel("Anomaly Score (max of combined heatmap)")
ax.set_ylabel("Number of files")
ax.set_title("Anomaly Score Distribution")
ax.legend()
ax.grid(True, alpha=0.3)

# Box plot
ax = axes[1]
bp = ax.boxplot([scores[~is_anom], scores[is_anom]], labels=["Normal", "Anomaly"],
                patch_artist=True)
bp["boxes"][0].set_facecolor("steelblue")
bp["boxes"][1].set_facecolor("crimson")
ax.set_ylabel("Anomaly Score")
ax.set_title("Normal vs Anomaly Scores")
ax.grid(True, alpha=0.3)

# Score rank plot
ax = axes[2]
sorted_scores = np.sort(scores)[::-1]
ax.plot(sorted_scores, marker=".", markersize=3, color="steelblue")
ax.axhline(global_threshold, color="red", linestyle="--", label=f"P95 threshold")
ax.fill_between(range(len(sorted_scores)), sorted_scores, global_threshold,
                 where=(sorted_scores > global_threshold), color="crimson", alpha=0.3,
                 label=f"Anomalous ({n_anomaly})")
ax.set_xlabel("File rank (highest score first)")
ax.set_ylabel("Anomaly Score")
ax.set_title("Ranked Anomaly Scores")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 6b. Dual Heatmap Visualization (Association × Reconstruction)

For the top anomalous file, visualize how the two signals combine:

In [ ]:
# Pick the most anomalous file
top = sorted_results[0]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
P = top["heatmap"].shape[1]            # number of patches (14)
patch_labels = [f"p{i}" for i in range(P)]

for ax, hm, title in zip(
    axes,
    [top["assoc_heatmap"], top["recon_heatmap"], top["heatmap"]],
    ["Association Discrepancy\n(low = anomalous)", "Reconstruction Error\n(high = anomalous)", "Combined\n(assoc × recon)"],
):
    im = ax.imshow(hm, aspect="auto", cmap="Reds", vmin=0)
    ax.set_xticks(range(P))
    ax.set_xticklabels(patch_labels, fontsize=8)
    ax.set_yticks(range(len(SENSOR_NAMES)))
    ax.set_yticklabels(SENSOR_NAMES, fontsize=9)
    ax.set_title(title, fontsize=10)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle(f"Top Anomaly: robot={top['robot_code']}, prog={top['program_number']}, "
             f"score={top['anomaly_score']:.4f}",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


### 6c. Counterfactual Visualization

Show what the model expected vs what actually happened on the most anomalous sensor-patch:

In [ ]:
# Find the most anomalous (sensor, patch) position in the top file
combined = top["heatmap"]  # [3, P]
max_si, max_pi = np.unravel_index(combined.argmax(), combined.shape)
print(f"Peak anomaly at sensor={SENSOR_NAMES[max_si]}, patch={max_pi}")

# Get the original raw window for this file
top_key = (top["robot_code"], str(top["start_time"]), top["program_number"])
top_rec = next(r for r in file_records if r["key"] == top_key)
raw_window = top_rec["window"]  # [3, 240]

# Compute patch time range
patch_size = config.patch_size   # 32
patch_stride = config.patch_stride  # 16
t_start = max_pi * patch_stride
t_end = min(t_start + patch_size, SEQ_LEN)

# Extract actual and use model to get predicted (counterfactual) patch
inp_t = torch.from_numpy(raw_window).unsqueeze(0).to(device)
robot_t = torch.tensor([top_rec["robot_idx"]], dtype=torch.long, device=device)
prog_t = torch.tensor([top_rec["program_idx"]], dtype=torch.long, device=device)

with torch.no_grad():
    tokens, local_patches = model._tokenize(inp_t, robot_t, prog_t)
    # Mask the single most anomalous token
    mask_single = torch.zeros(1, tokens.shape[1], dtype=torch.bool, device=device)
    token_idx = max_si * P + max_pi
    mask_single[0, token_idx] = True
    tokens_masked = model.mask_token(tokens, mask_single)
    out_t = model.transformer.forward_with_detach_mode(tokens_masked, detach="none")
    predicted_patch = model.decoder(out_t["out"])[0, token_idx].cpu().numpy()  # [32]

# Actual local patch (RevIN'd)
actual_patch = local_patches[0, token_idx].cpu().numpy()  # [32]

fig, ax = plt.subplots(figsize=(12, 4))
time_axis = np.arange(t_start, t_end)
ax.plot(time_axis, actual_patch, label="Actual (local RevIN'd)", color="crimson", linewidth=2)
ax.plot(time_axis, predicted_patch, label="Model prediction (counterfactual normal)", color="steelblue", linewidth=2, linestyle="--")
ax.fill_between(time_axis, actual_patch, predicted_patch,
                alpha=0.3, color="crimson", label="Anomaly gap")
ax.set_xlabel("Timestep")
ax.set_ylabel("Normalized sensor value")
ax.set_title(f"Counterfactual: {SENSOR_NAMES[max_si]} at patch {max_pi} "
             f"(t={t_start}–{t_end})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print(f"Gap (MSE): {((actual_patch - predicted_patch) ** 2).mean():.6f}")


## 7. Summary Report

Export results and compare normal vs anomaly profiles.

In [ ]:
print("=" * 70)
print("MASKFORMER INFERENCE SUMMARY")
print("=" * 70)
print(f"  Model:              {model.name}")
print(f"  Checkpoint:         {CKPT_PATH}")
print(f"  Device:             {device}")
print(f"  Total files scored: {len(results)}")
print(f"  Threshold (P95):    {global_threshold:.6f}")
print(f"  Normal files:       {n_normal} ({100*n_normal/len(results):.1f}%)")
print(f"  Anomalous files:    {n_anomaly} ({100*n_anomaly/len(results):.1f}%)")
print()
print("Per-sensor thresholds (P95):")
for sn in SENSOR_NAMES:
    print(f"  {sn}: {sensor_thresholds[sn]:.6f}")
print()
print("Top-3 anomalous files:")
for i, r in enumerate(sorted_results[:3]):
    sensors_flagged = [sn for sn in SENSOR_NAMES if r["sensor_verdicts"][sn]]
    print(f"  #{i+1}: robot={r['robot_code']}, prog={r['program_number']}, "
          f"score={r['anomaly_score']:.6f}, sensors={sensors_flagged}")
print("=" * 70)
